In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
DISTRIBUTION = "HIERARCHICAL_PAIRS"

In [ ]:
df = pd.read_parquet("data/mp_targeted/results.parquet")

In [ ]:
# Best F1 per benchmark
for bench in df.index.get_level_values("benchmark").unique():
    group = df.loc[bench]
    best = group.loc[group["f1_score"].idxmax()]
    print(f"{bench}: F1={best['f1_score']:.4f} ")

In [ ]:
df.loc[DISTRIBUTION]

## Focus distribution

In [ ]:
sub = df.loc[DISTRIBUTION].reset_index()
sub.head()

## Marginal effects on F1 score

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["By Regime", "By SAE L0 (effective sparsity)"],
)

fig.add_trace(
    go.Box(
        x=sub["regime"],
        y=sub["f1_score"],
        boxpoints="all",
        jitter=0.3,
        pointpos=0,
        marker_color=px.colors.qualitative.Plotly[0],
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=sub["sae_l0"],
        y=sub["f1_score"],
        mode="markers+text",
        text=sub["regime"].str.replace("_l0", ""),
        textposition="top center",
        marker=dict(size=10, color=px.colors.qualitative.Plotly[1]),
    ),
    row=1,
    col=2,
)

fig.update_layout(
    height=450,
    width=900,
    title_text=f"F1 Score by Regime and Effective Sparsity ({DISTRIBUTION})",
    showlegend=False,
)
fig.show()

## F1 & MCC vs sae_l0 by regime

In [ ]:
melted = sub.melt(
    id_vars=["sae_l0", "regime"],
    value_vars=["f1_score", "mcc"],
    var_name="metric",
    value_name="score",
)

fig = px.scatter(
    melted,
    x="sae_l0",
    y="score",
    color="regime",
    facet_col="metric",
    category_orders={"metric": ["f1_score", "mcc"]},
    labels={
        "sae_l0": "SAE L0",
        "score": "Score",
        "regime": "Regime",
    },
    title=f"F1 & MCC vs SAE L0 by regime ({DISTRIBUTION})",
    height=400,
    width=900,
)
# Add true_l0 reference line
for i in range(1, 3):
    fig.add_vline(
        x=sub["true_l0"].mean(),
        line_dash="dash",
        line_color="gray",
        annotation_text="true L0",
        row=1,
        col=i,
    )
fig.update_traces(marker_size=10)
fig.show()

## Precision vs Recall by regime

In [ ]:
fig = px.scatter(
    sub,
    x="recall",
    y="precision",
    color="regime",
    size="sae_l0",
    hover_data=["max_iterations", "residual_threshold", "sae_l0", "f1_score"],
    labels={
        "precision": "Precision",
        "recall": "Recall",
        "regime": "Regime",
        "sae_l0": "SAE L0",
    },
    title=f"Precision vs Recall by regime (size = SAE L0) ({DISTRIBUTION})",
    height=450,
    width=700,
)
fig.update_traces(marker=dict(sizemin=5))
fig.show()

## F1 & MCC vs max_iterations (fixed_l0 regime)

In [ ]:
fixed = sub[sub["regime"] == "fixed_l0"].copy()

melted = fixed.melt(
    id_vars=["max_iterations"],
    value_vars=["f1_score", "mcc"],
    var_name="metric",
    value_name="score",
)

fig = px.line(
    melted,
    x="max_iterations",
    y="score",
    color="metric",
    facet_col="metric",
    category_orders={"metric": ["f1_score", "mcc"]},
    markers=True,
    labels={
        "max_iterations": "Max Iterations",
        "score": "Score",
        "metric": "Metric",
    },
    title=f"F1 & MCC vs Max Iterations — fixed_l0 regime ({DISTRIBUTION})",
    height=400,
    width=900,
)
fig.add_vline(
    x=fixed["true_l0"].mean(),
    line_dash="dash",
    line_color="gray",
    annotation_text="true L0",
)
fig.update_layout(showlegend=False)
fig.show()